In [ ]:
# installation des programmes
!pip install unidecode
!pip install selenium
!pip install ydata_profiling

In [ ]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver import ChromeOptions
from datetime import datetime
from unidecode import unidecode
from ast import literal_eval
from itertools import chain
from ydata_profiling import ProfileReport

In [ ]:
df = pd.read_excel("/content/Adresses_Nouvelle_Caledonie.xlsx")

In [ ]:
df.columns = ["Ville", "Commune", "Adresse", "CodePostale"]
df

In [ ]:
df['Commune'].unique()

In [ ]:
# Dictionnaire mapping : commune/quartier → code postal
postal_map = {
    # --- Dumbéa (98835) ---
    'Les Koghis': "98835",
    'Auteuil': "98835",
    'Katiramona sud': "98835",
    'Nondoué': "98835",
    'Koutio': "98835",
    'Dumbéa-sur-Mer': "98835",
    'Couvelée': "98835",
    'Plaine de Koé': "98835",
    'Coeur de ville': "98835",
    'Plaine Adam': "98835",
    'Nakutakoin': "98835",
    "La Plaine": "98835",

    # --- Mont-Dore (98809 / 98810 / 98876) ---
    'PLUM': "98810",
    'Plum': "98810",
    'MONT-DORE SUD': "98809",
    'Mont-Dore Sud': "98810",
    'Mont-Dore': "98810",
    'Mont-Dore Nord': "98810",
    'Mont-Dore Est': "98810",
    'Mont-Dore Ouest': "98810",
    'VALLON-DORE': "98809",
    'LA COULEE': "98876",
    'La Coulée': "98810",
    'SAINT-LOUIS': "98809",
    'YAHOUE': "98809",
    'ROBINSON': "98809",
    'SAINT-MICHEL': "98809",
    'BOULARI': "98809",
    'LA LEMBI-MOUIRANGE': "98809",
    'LE GRAND SUD': "98809",
    'LA CONCEPTION': "98809",
    'PONT DES FRANCAIS': "98809",
    'Pont des Francais': "98810",
    'La Couronne': "98810",

    # --- Nouméa (98800) ---
    'HAUT-MAGENTA': "98800",
    'ARTILLERIE': "98800",
    'VAL PLAISANCE': "98800",
    'VALLEE DES COLONS': "98800",
    'RIVIERE SALEE': "98800",
    'BAIE DES CITRONS': "98800",
    'Baie des Citrons': "98800",
    'AERODROME': "98800",
    'PORTES DE FER': "98800",
    'Portes de Fer': "98800",
    'VALLEE DU GENIE': "98800",
    'RECEIVING': "98800",
    'ORPHELINAT': "98800",
    'ANSE VATA': "98800",
    'Anse Vata': "98800",
    'SEPTIEME KILOMETRE': "98800",
    'TRIANON': "98800",
    'DUCOS INDUSTRIEL': "98800",
    'DONIAMBO': "98800",
    'VALLEE DU TIR': "98800",
    'MOTOR POOL': "98800",
    'MONTRAVEL': "98800",
    'QUARTIER LATIN': "98800",
    'NUMBO': "98800",
    'LOGICOOP': "98800",
    'DUCOS': "98800",
    "N'GEA": "98800",
    'SIXIEME KILOMETRE': "98800",
    'MAGENTA': "98800",
    'Faubourg Blanchot': "98800",
    'FAUBOURG BLANCHOT': "98800",
    'NORMANDIE': "98800",
    'OUEMO': "98800",
    'Ouémo': "98800",
    'NOUVILLE': "98800",
    'QUATRIEME KILOMETRE': "98800",
    'CENTRE VILLE': "98800",
    'Centre Ville': "98800",
    'TINA': "98800",
    'Tina': "98800",
    'TINDU': "98800",
    'MONTAGNE COUPEE': "98800",
    'KAMERE': "98800",

    # --- Koné / Kaala-Gomen / Pouembout ---
    'KONE': "98860",
    'Koné': "98860",
    'KOUMOUROU': "98845",
    'POUEMBOUT': "98825",
    'Pouembout': "98825",
    'KAALAGOMEN': "98817",
    'KaalaGomen': "98817",
}

# Dictionnaire mapping : ville → code postal
ville_to_postal = {
    "MONTDORE": 98810,
    "MONT-DORE": 98810,
    "NOUMEA": 98800,
    "DUMBEA": 98835,
    "PAITA": 98890
}

In [ ]:
# Appliquer le mapping
df2 = df
df2['CodePostale'] = df2["Commune"].map(postal_map)
df2.loc[[9886], ['CodePostale']] = '98810'
df2.loc[[18818], ['CodePostale']] = '98810'
df2.loc[[29242], ['CodePostale']] = '98810'
df2.loc[[29286], ['CodePostale']] = '98800'

In [ ]:
df2

In [ ]:
# Pour repérer les communes non mappées (code postal manquant)
missing = df2[df2["CodePostale"].isna()]["Commune"].unique()
print("Communes sans code postal assigné :", missing)

In [ ]:
df2[df2.isnull().any(axis=1)]

In [ ]:
df2['Commune'].unique()

Telecharge le dataframe 2 au format excel

In [ ]:
output_file = "df2_export.xlsx"
df2.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)

In [ ]:
df = pd.read_excel("/content/Adresses_Nouvelle-Caledonie.xlsx (à importer).xlsx")
df = df.rename(columns={
    'Ville (Ville)': 'Ville',
    'Quartier (Quartier)': 'Quartier',
    'Adresse (Adresse)': 'Adresse',
    'Code Postal (CodePostal)': 'Code Postale'})

In [ ]:
#recupérer les villes
arr = df['Ville'].unique()
dfV = pd.DataFrame(arr, columns=['Ville'])
output_file = "V_NC.xlsx"
dfV.to_excel(output_file, index=False)
from google.colab import files
files.download(output_file)

In [ ]:
#recupérer les quartiers
arr = df['Quartier'].unique()
dfQ = pd.DataFrame(arr, columns=['Quartier'])
output_file = "Q_NC.xlsx"
dfQ.to_excel(output_file, index=False)
files.download(output_file)

In [ ]:
#récuperer les codes postaux
arr = df['Code Postale'].unique()
dfC = pd.DataFrame(arr, columns=['Code Postale'])
output_file = "C_NC.xlsx"
dfC.to_excel(output_file, index=False)
files.download(output_file)